In [1]:
import os
import uuid
import hashlib
import platform
import numpy as np
import scipy
import matplotlib
import pandas as pd
# 修改为自己的学号
STUDENT_ID = 20244080128
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")


[Conda 环境] env_20244080128
[学号]       20244080128
[机器指纹] e28ba7c1e7aa42dd
[RUN_ID]   20244080128-e28ba7c1e7aa42dd


# 实验一

## 1

In [2]:
import numpy as np

# 1) 创建三种类型的数组
arr_int   = np.array([1, 2, 3, 4])
arr_float = np.array([1.5, 2.5, 3.5, 4.5])
arr_bool  = np.array([True, False, True, False])

# 2) 用循环批量输出属性，代码更简洁
for name, arr in [("整数", arr_int), ("浮点", arr_float), ("布尔", arr_bool)]:
    print(f"[{name}数组] dtype={arr.dtype}, shape={arr.shape}, ndim={arr.ndim}")

[整数数组] dtype=int64, shape=(4,), ndim=1
[浮点数组] dtype=float64, shape=(4,), ndim=1
[布尔数组] dtype=bool, shape=(4,), ndim=1


## 2

**改变形状的条件：**
1. 原数组元素总数 `size = 12`，目标形状元素总数必须等于 12：
   - `(3,4)`：3×4 = 12 ✅
   - `(2,3,2)`：2×3×2 = 12 ✅
   - `(5,3)`：5×3 = 15 ≠ 12 ❌ → 触发 ValueError（如上"预期报错"所示）。
2. 元素按行优先顺序（C 序）依次填入新形状：
   如 arr_34 第1行 0~3、第2行 4~7、第3行 8~11。
3. reshape 返回新视图，不修改原 arr_tt（原数组仍为长度12的一维数组）。

In [10]:
import numpy as np

arr_tt = np.arange(12)
print("arr_tt =\n", arr_tt, "\nshape =", arr_tt.shape)

arr_34 = arr_tt.reshape(3, 4)
print("\narr_34 =\n", arr_34, "\nshape =", arr_34.shape)

arr_232 = arr_tt.reshape(2, 3, 2)
print("\narr_232 =\n", arr_232, "\nshape =", arr_232.shape)

# 验证：元素总数不相等时 reshape 报错
try:
    arr_tt.reshape(5, 3)
except ValueError as e:
    print("\n【预期报错】元素总数不等，无法 reshape：", e)

arr_tt =
 [ 0  1  2  3  4  5  6  7  8  9 10 11] 
shape = (12,)

arr_34 =
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]] 
shape = (3, 4)

arr_232 =
 [[[ 0  1]
  [ 2  3]
  [ 4  5]]

 [[ 6  7]
  [ 8  9]
  [10 11]]] 
shape = (2, 3, 2)

【预期报错】元素总数不等，无法 reshape： cannot reshape array of size 12 into shape (5,3)


## 3

**结果解释：**

1. **取第2行（`arr_34[1, :]`）**：结果为 `[4 5 6 7]`，形状为 `(4,)`。
   - 原理：使用了整数下标 `1`，这代表“抽走”这一维，因此数组从二维降为了一维。

2. **取第1至第3列（`arr_34[:, 0:3]`）**：结果为 3×3 的矩阵，形状为 `(3, 3)`。
   - 原理：使用了切片 `0:3`（左闭右开，即索引 0,1,2 列）。切片代表“框选范围”，因此保留了原有的二维结构。

3. **取大于6的元素（`arr_34[mask]`）**：结果为 `[7 8 9 10 11]`，形状为 `(5,)`。
   - 原理：`mask` 是一个与原数组形状相同的布尔矩阵。布尔索引会扫描所有位置，把对应为 `True` 位置的元素挑出来并排成一条线。因此，不论原数组是几维，布尔索引的结果**始终是一维数组**，其长度等于 `True` 的个数（即 5 个）。

In [16]:
arr_34_h = arr_34[1,:]
arr_34_l = arr_34[:,0:3]
print(arr_34_h)
print(arr_34_l)
mask = arr_34 > 6
arr_34_t = arr_34[mask]
print(arr_34_t)
print(arr_34_t.shape)

[4 5 6 7]
[[ 0  1  2]
 [ 4  5  6]
 [ 8  9 10]]
[ 7  8  9 10 11]
(5,)


## 4

In [18]:
b = np.arange(10)
mask = (b > 7) | (b < 2)
b_r = b[mask]
print(b_r)
mask = (b > 3) & (b < 7)
b_t = b[mask]
print(b_t)

[0 1 8 9]
[4 5 6]


## 5

**结果解释与区别说明：**

1. **逐元素乘法（`*` 与 `np.multiply`）**：
   - 运算规则：两个矩阵对应位置的元素直接相乘，即 \(C_{ij} = A_{ij} \times B_{ij}\)。数学上称为 Hadamard 积。
   - 结果矩阵的形状与原矩阵保持一致（本题为 2×2）。
   - 例如左上角元素为：\(1 \times 5 = 5\)。

2. **矩阵乘法（`@` 与 `np.matmul`）**：
   - 运算规则：左矩阵的“行”与右矩阵的“列”对应元素相乘后求和（行乘列再相加）。即 \(C_{ij} = \sum A_{ik} \times B_{kj}\)。这是标准的线性代数矩阵乘法。
   - 结果矩阵的形状由两个矩阵的维度决定（(m×n) 乘 (n×p) 得到 m×p，本题 2×2 乘 2×2 得到 2×2）。
   - 例如左上角元素为：\(A\) 的第 1 行 \([1, 2]\) 与 \(B\) 的第 1 列 \([5, 7]\) 的点乘，即 \(1 \times 5 + 2 \times 7 = 19\)。

3. **总结**：`*` 是位置对齐的纯数值乘法，`@` 是带有线性变换意义的代数乘法。在 NumPy 中，两者不能混用。

In [20]:
import numpy as np

# 构造两个 2x2 数组
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

print("矩阵 A =\n", A)
print("矩阵 B =\n", B)

# 1. 逐元素乘法
print("\n--- 逐元素乘法 ---")
mul_1 = A * B
mul_2 = np.multiply(A, B)
print("A * B =\n", mul_1)
print("np.multiply(A, B) =\n", mul_2)

# 2. 矩阵乘法
print("\n--- 矩阵乘法 ---")
mat_1 = A @ B
mat_2 = np.matmul(A, B)
print("A @ B =\n", mat_1)
print("np.matmul(A, B) =\n", mat_2)

矩阵 A =
 [[1 2]
 [3 4]]
矩阵 B =
 [[5 6]
 [7 8]]

--- 逐元素乘法 ---
A * B =
 [[ 5 12]
 [21 32]]
np.multiply(A, B) =
 [[ 5 12]
 [21 32]]

--- 矩阵乘法 ---
A @ B =
 [[19 22]
 [43 50]]
np.matmul(A, B) =
 [[19 22]
 [43 50]]


# 实验二

## 1

**结果解释：**
1. **固定随机种子**：使用 `np.random.seed(42)` 确保了每次运行代码生成的伪随机数序列完全一致，保证了实验的**可复现性**。
2. **截断处理**：理论上的正态分布是没有上下限的，但在现实中（如成绩）存在 0~100 的边界。`np.clip(scores, 0, 100)` 将所有小于 0 的值变成了 0，大于 100 的值变成了 100。
3. **统计指标**：均值（70.18）和标准差（9.75）都非常接近我们设定的理论值（70 和 10）。最大值变成了 100，说明 `np.clip` 成功拦截了部分超出的异常值。

In [5]:
import numpy as np
np.random.seed(42)
scores = np.random.normal(loc = 70,scale = 10, size = 1000)
scores = np.clip(scores, 0, 100)

print("样本数：", len(scores))
print("最小值：", scores.min())
print("最大值：", scores.max())
print("均值：", scores.mean())
print("标准差：", scores.std())
print("极差：", scores.max() - scores.min())

样本数： 1000
最小值： 37.58732659930927
最大值： 100.0
均值： 70.18400443523215
标准差： 9.755086339034765
极差： 62.41267340069073


## 2

**结果解释与抽样对比：**
1. **均匀抽样**：不使用 `p` 参数时，默认每个元素被抽中的概率完全相等（均为 1/10）。输出结果 `[3 2 5 6 1]` 中大小数字分布均匀。
2. **加权抽样**：通过 `p=weights` 传入了概率数组，数字越大（如 10、9、8），其对应的概率值越大（概率为 \(i/55\)）。输出结果 `[4 1 8 10 9]` 明显更容易抽中大数字。
3. **不放回**：两次抽样均设置了 `replace=False`，保证了同一个数字不会在单次抽样中重复出现。在加权且不放回的情况下，NumPy 会自动把抽走元素的概率剔除并对剩余概率重新归一化。

In [27]:
arr = np.arange(1, 11)  # [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
uniform_sample = np.random.choice(arr, size=5, replace=False)
print(uniform_sample)
# 权重直接用数字本身，但是要除以总和，让它变成概率
weights = arr / arr.sum()
# 或者用 np.ones(10) 也可以，但题目要求"按整数大小成比例"
weighted_sample = np.random.choice(arr, size=5, replace=False, p=weights)
print(weighted_sample)

[ 4 10  6  3  7]
[ 6 10  7  9  3]


## 3

**结果解释与区别：**
1. **`np.random.permutation(arr)`**：用于生成一个随机排列的新数组。从输出可以看出，它**不修改原数组**，原数组 `arr` 依然保持 `[1, 2, 3, ..., 10]` 的原始顺序，打乱后的结果是一个独立的副本。
2. **`np.random.shuffle(arr)`**：用于**原地打乱（In-place）**数组。从输出可以看出，它的返回值为 `None`，因为它直接在原数组的内存上进行修改，打印原数组时发现 `arr` 的顺序已经被彻底打乱了。
3. **使用建议**：如果不希望破坏原始数据，用 `permutation`；如果数据量极大且不需要保留原顺序，用 `shuffle` 可以节省内存。

In [ ]:
# 先定义一个用来测试的数组
arr = np.arange(1, 11)
print("原始数组：", arr)

# 1. 用 permutation 随机排列
print("\n--- np.random.permutation ---")
perm_res = np.random.permutation(arr)
print("permutation 返回的新数组：", perm_res)
print("打印原数组 (看看变没变)：", arr)

# 2. 用 shuffle 原地打乱
print("\n--- np.random.shuffle ---")
shuffle_res = np.random.shuffle(arr)
print("shuffle 的返回值：", shuffle_res)
print("打印原数组 (看看变没变)：", arr)

原始数组： [ 1  2  3  4  5  6  7  8  9 10]

--- np.random.permutation ---
permutation 返回的新数组： [10  5  2  6  8  4  7  3  1  9]
打印原数组 (看看变没变)： [ 1  2  3  4  5  6  7  8  9 10]

--- np.random.shuffle ---
shuffle 的返回值： None
打印原数组 (看看变没变)： [ 2  9  3  1  8  4  6  5  7 10]


## 4

**结果解释：**
1. **普通均值为 `nan`**：在 NumPy 中，`np.nan`（Not a Number）具有“传染性”。任何数与 `nan` 进行算术运算（如加法、求平均），结果都会变成 `nan`。因此调用普通的 `scores_nan.mean()` 直接得到了 `nan`。
2. **忽略缺失值的均值**：使用 `np.nanmean()` 函数会自动跳过所有的 `nan` 值，只对剩下的 997 个有效样本求均值（算得约 69.42，非常接近理论均值 70），这才是真实有效的统计量。
3. **缺失值统计**：`np.isnan(scores_nan)` 会返回一个布尔数组（`True` 代表是缺失值），利用布尔值 `True` 会被当作 1、`False` 当作 0 的特性，对其调用 `.sum()` 即可快速统计出缺失值个数为 3 个。

**拓展**：除了 `np.nanmean`，NumPy 还提供了 `np.nansum`、`np.nanstd`、`np.nanmax` 等一系列以 `nan` 开头的函数，专门用于忽略缺失值进行统计计算。

In [58]:

# 1. 生成数据
np.random.seed(42)
scores_nan = np.random.normal(loc=70, scale=10, size=1000)
scores_nan = np.clip(scores_nan, 0, 100)

# 2. 设置缺失值
scores_nan[9] = np.nan
scores_nan[19] = np.nan
scores_nan[29] = np.nan

# 3. 计算均值
print("普通均值：", scores_nan.mean())
scores_nan_c = np.nanmean(scores_nan)
print("忽略缺失值的均值：", scores_nan_c)

# 4. 统计缺失值个数
print("缺失值个数：", np.isnan(scores_nan).sum())


普通均值： nan
忽略缺失值的均值： 70.19620743160239
缺失值个数： 3
